In [34]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator #Genera datos (imagenes) sintéticos
from keras import optimizers #Técnica del descenso del gradiente, sirve para minimizar el error en el proceso de entrenamiento
from keras.models import Sequential # Estacebe un modelo de red neuronal por capas
from keras.layers import Dense, Flatten, Dropout, Activation
#Dense = define las neuronas por cada capa
#Flatter = aplana los datos en formato matriz N-Tensor --> 1-Tensor
#Dropout = técnica de reducción del sobreajuste o sobre entrenamiento (apagar % de neuronas)
#Activation = determina las funciones de activiación en cada neurona (Relu, sigmoid, softmax, tanh, linear, etc)
from keras.layers import Convolution2D, MaxPooling2D


In [ ]:
#Definir los hiperparametros de la red neurona convolucional (Convolutional Neural Network - CNN)

#Definir la ruta de los datos de entranamiento
entrenar = "CNN_Imagenes/entrenar"
validar = "CNN_Imagenes/validar"

#Hiperparametros
epocas = 100
altura,anchura = 400,400
batch_size = 2
pasos = 100

#Definir la cantidad de kernels por cada capa
kernel1=32 # 2,4,8,16,32,64,128,256, 512, etc
kernel1_size = (3,3)
kernel2=64
kernel2_size = (4,4)
size_pooling = (3,3)
clases = 2 #Numero de objetos a detectar o identificar




In [36]:
#Generar datos sintéticos (se recomienda si la cantidad de datos es pequeña)

entrenamiento = ImageDataGenerator(rescale=1/255,
                             zoom_range=0.2,
                             horizontal_flip=True)

validacion = ImageDataGenerator(rescale=1/255)

#Extraer las imagenes de las carpetas

imagenes_entrenamiento = entrenamiento.flow_from_directory(entrenar,
                                                      target_size=(anchura,altura),
                                                      batch_size=batch_size,
                                                      class_mode="categorical")
imagenes_validacion = validacion.flow_from_directory(validar,
                                                  target_size=(anchura,altura),
                                                  batch_size=batch_size,
                                                  class_mode="categorical")

Found 1000 images belonging to 2 classes.
Found 300 images belonging to 2 classes.


In [38]:
#Definir la arquitectura de la red neuronal convolucional

CNN = Sequential()

CNN.add(Convolution2D(kernel1,
                      kernel1_size,
                      padding="same",
                      input_shape=(altura,anchura,3),
                      activation="relu")) #Primera capa convolucional
CNN.add(MaxPooling2D(pool_size=size_pooling)) #Capa de submuestreo 

CNN.add(Convolution2D(kernel2,
                      kernel2_size,
                      padding="same",
                      input_shape=(altura,anchura,3),
                      activation="relu")) #Segunda capa convolucional
CNN.add(MaxPooling2D(pool_size=size_pooling)) #Capa de submuestreo 

#Aplanar las matrices en formato de vector
CNN.add(Flatten())

#Conectar a el perceptrón multicapa
CNN.add(Dense(255,activation="relu"))
CNN.add(Dense(255,activation="relu"))
CNN.add(Dense(255,activation="relu"))
CNN.add(Dropout(0.5))

#Definir la capa de salida
CNN.add(Dense(clases,activation="softmax"))


In [ ]:
#Definir los parametros del entrenamiento
CNN.compile(loss="categorical_crossentropy",optimizer="adam",metrics=["acc","mse"])

In [40]:
#Realizamos el entrenamiento
historico = CNN.fit(imagenes_entrenamiento,
                    validation_data=imagenes_validacion,
                    epochs=epocas,
                    validation_steps=pasos,
                    verbose=1)

Epoch 1/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 89s 173ms/step - acc: 0.5740 - loss: 0.7227 - mse: 0.2365 - val_acc: 0.6550 - val_loss: 0.6515 - val_mse: 0.2302
Epoch 2/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 102s 203ms/step - acc: 0.6330 - loss: 0.6604 - mse: 0.2188 - val_acc: 0.5000 - val_loss: 6.1755 - val_mse: 0.4998
Epoch 3/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 117s 233ms/step - acc: 0.6820 - loss: 0.6401 - mse: 0.1987 - val_acc: 0.6250 - val_loss: 0.9015 - val_mse: 0.2431
Epoch 4/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 153s 306ms/step - acc: 0.7840 - loss: 0.5187 - mse: 0.1531 - val_acc: 0.5850 - val_loss: 0.8848 - val_mse: 0.2591
Epoch 5/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 170s 339ms/step - acc: 0.8480 - loss: 0.3648 - mse: 0.1119 - val_acc: 0.7700 - val_loss: 0.6480 - val_mse: 0.1666
Epoch 6/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 179s 357ms/step - acc: 0.8550 - loss: 0.3525 - mse: 0.1070 - val_acc: 0.7900 - val_loss: 0.5175 - val_mse: 0.1555
Epoch 7/100
500/500 ━━━━━━━━━━━━━━━━━━━━ 182s 363ms/step - acc: 0.8780 

In [41]:
#Guarda el modelo entrenado
CNN.save("CNN_Imagenes/Modelo/cnn.h5")
CNN.save_weights("CNN_Imagenes/Modelo/cnn_pesos.weights.h5")

In [ ]:
# Evaluando el modelo entrenado con LÓGICA DE DECISIÓN CNN + OCR

import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array
from keras.models import load_model
import cv2
import easyocr
import ssl
import warnings

warnings.filterwarnings('ignore')
ssl._create_default_https_context = ssl._create_unverified_context

# 1. Configuracion
ruta_imagen = "CNN_Imagenes/validar/negativos/BikesHelmets69_png.rf.e2da0ac295036cebbb755057a2d5ad5d.jpg"
altura, anchura = 400, 400 
modelo_path = "CNN_Imagenes/Modelo/cnn.h5"
pesos_path = "CNN_Imagenes/Modelo/cnn_pesos.weights.h5"

# 2. Cargar modelos
print("Analizando la imagen...")
reader = easyocr.Reader(['en'], gpu=False, verbose=False) 
cnn = load_model(modelo_path, compile=False)
cnn.load_weights(pesos_path)

if os.path.exists(ruta_imagen):
    # 3. Prediccion de la CNN
    img_cnn = load_img(ruta_imagen, target_size=(anchura, altura))
    img_cnn = img_to_array(img_cnn) / 255.0
    img_cnn = np.expand_dims(img_cnn, axis=0)
    clase_pred = cnn.predict(img_cnn, verbose=0)
    arg_max = np.argmax(clase_pred[0])

    print("\n" + "="*40)
    print(f"ANALISIS DE LA CNN:")
    print("="*40)

    if arg_max == 1: 
        print("ESTADO: PLACA DETECTADA. Iniciando lectura OCR...")
        
        # 4. Solo si la CNN aprobo
        img_cv = cv2.imread(ruta_imagen)
        img_res = cv2.resize(img_cv, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
        gray = cv2.cvtColor(img_res, cv2.COLOR_BGR2GRAY)
        resultados = reader.readtext(gray)
        
        if not resultados:
            print("RESULTADO: No se pudo leer el texto claramente.")
        else:
            for (bbox, texto, probabilidad) in resultados:
                if probabilidad > 0.15:
                    print(f"MATRÍCULA: {texto.upper()}")
    else:
        print("ESTADO: PLACA NO DETECTADA. Lectura cancelada.")
    
    print("="*40)
else:
    print(f"Error: No existe el archivo {ruta_imagen}")

Cargando inteligencia...

ANÁLISIS DE LA IA
ESTADO: Esto no parece una placa. Lectura cancelada.


In [ ]:
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array
from keras.models import load_model
import os.path
import cv2
import easyocr
import ssl

# Configuracion para Mac
ssl._create_default_https_context = ssl._create_unverified_context

# Inicializar el lector fuera de la función para mayor velocidad
reader = easyocr.Reader(['en'], gpu=False, verbose=False)

def evaluar(imagen):
    # Valores de los hiperparametros
    altura, anchura = 400, 400 
    modelo = "CNN_Imagenes/Modelo/cnn.h5"
    pesos = "CNN_Imagenes/Modelo/cnn_pesos.weights.h5"

    # 1. Cargar modelo
    cnn = load_model(modelo, compile=False)
    cnn.load_weights(pesos)

    # 2. Preprocesar para la CNN
    imagen_pre = cv2.resize(imagen, (anchura, altura))
    imagen_pre = imagen_pre / 255.0
    imagen_pre = img_to_array(imagen_pre)
    imagen_pre = np.expand_dims(imagen_pre, axis=0)

    # 3. Prediccion
    clase = cnn.predict(imagen_pre, verbose=0)
    arg_max = np.argmax(clase[0])

    print("\n" + "="*40)
    print("ANALISIS DE LA CNN:")
    print("="*40)
    
    # --- LÓGICA DE DECISIÓN ---
    if arg_max == 1: 
        print("ESTADO: PLACA DETECTADA. Iniciando lectura OCR...")
        
        # 4. OCR con Zoom para mejorar lectura de placas
        img_zoom = cv2.resize(imagen, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
        gray = cv2.cvtColor(img_zoom, cv2.COLOR_BGR2GRAY)
        resultados_ocr = reader.readtext(gray)

        if not resultados_ocr:
            print("RESULTADO: No se pudo leer el texto claramente.")
        else:
            for (bbox, texto, probabilidad) in resultados_ocr:
                if probabilidad > 0.20: # Sensibilidad media para cAmara en vivo
                    print(f"MATRICULA: {texto.upper()}")
    elif arg_max == 0:
        print("ESTADO: PLACA NO DETECTADA. Pudo haberse confundido y haber capturado otra cosa :))) xdxdxd")
    
        
    print("="*40)

In [47]:
import cv2

capture = cv2.VideoCapture(0)

while(1):
    _,frame = capture.read() #leemos cada frame de la camara
    cv2.imshow("Ventana", frame)
    c = cv2.waitKey(5) & 0xFF #Esperamos una tecla

    if (c==27):
        break

    if (c==99):
        evaluar(frame)

capture.release()
cv2.destroyAllWindows()
cv2.waitKey(1) # Pequeño truco para Mac: fuerza el cierre de la ventana
cv2.waitKey(1)
cv2.waitKey(1)
cv2.waitKey(1)


ANALISIS DE LA CNN:
ESTADO: PLACA NO DETECTADA. Puede haberse confundido y haber capturado una persona, calle, letrero, bicicleta o carro sin placas.

ANALISIS DE LA CNN:
ESTADO: PLACA NO DETECTADA. Puede haberse confundido y haber capturado una persona, calle, letrero, bicicleta o carro sin placas.

ANALISIS DE LA CNN:
ESTADO: PLACA NO DETECTADA. Puede haberse confundido y haber capturado una persona, calle, letrero, bicicleta o carro sin placas.


-1